# Installation/Setup
- Installs DSSP, Clustal Omega, and MUSCLE (structure/alignment tools the pipeline calls out to)
- Clones the BE3D package + example scripts/data and installs it (and its Python dependencies) into this Colab runtime
- Configures Plotly's renderer for Colab and enables the ipywidgets custom widget manager, both required for the interactive plots below to actually display


In [ ]:
# @title Install DSSP, ClustalO, MUSCLE
! apt-get -qq update && apt-get -qq install -y dssp clustalo
! wget -q https://github.com/rcedgar/muscle/releases/download/v5.3/muscle-linux-x86.v5.3 -O /content/muscle
! chmod +x /content/muscle

# @title Clone BE3D (pipeline package + example scripts + example data)
# @markdown Branch used below is a working branch, not main -- repoint at main once merged.
BECLUST3D_BRANCH = "feature/optimize-colab-notebooks"
! rm -rf /content/beclust3d-public
! git clone --quiet --branch {BECLUST3D_BRANCH} --depth 1 https://github.com/broadinstitute/BE3D.git /content/beclust3d-public

# @title Install Python dependencies
# be3d_local.py runs as its own subprocess (sys.executable) importing the full beclust3d
# package, not just the plotting helpers imported directly below -- installing the
# cloned package itself (rather than hand-listing packages here) pulls in everything
# pyproject.toml declares (numpy, biopython, biopandas, DSSPparser, wget, etc.), so this
# stays in sync with the package's real dependencies instead of drifting from them.
! pip install -q /content/beclust3d-public
! pip install -q ipymolstar

import os
import sys
import subprocess
import copy
import yaml
import pandas as pd
import numpy as np
import plotly.io as pio
from IPython.display import display, Image, SVG
from ipywidgets import interact, interact_manual, Dropdown
from ipymolstar import PDBeMolstar

# Colab requires its custom widget manager to be explicitly enabled for ipywidgets
# Output widgets (as used by interact()/show_picker()) to render correctly. Without
# this, anything display()'d from inside an interact() callback -- including every
# Plotly figure in the QA, BE-Clust3D, PPI, PPI-MetaClust3D, merged-results, and
# blind_target sections below -- renders blank, since it never reaches Colab's normal
# cell-output pipeline. (Monomer BE-MetaClust3D was the one section unaffected by this,
# since it has no per-screen dropdown and never calls interact().)
from google.colab import output as _colab_output
_colab_output.enable_custom_widget_manager()

# Colab's own Plotly renderer -- see BE3D_local.ipynb's Setup cell for why this matters:
# leaving pio.renderers.default on whatever got auto-detected can resolve to MULTIPLE
# renderers at once (e.g. "colab+notebook_connected"), so every fig.show() call renders
# once per registered renderer, stacking visible duplicates of every single plot.
pio.renderers.default = 'colab'

# Colab's 'colab' renderer loads its JS bundle asynchronously the FIRST time a figure is
# shown in a given session. A display()/show() call that fires before that finishes loading
# renders a permanently blank output (it doesn't retry) -- later figures then render fine
# once the bundle is cached. Warming it up here with a throwaway figure, before any real
# plot runs, avoids ever hitting that race on a plot that actually matters.
import plotly.graph_objects as _go
display(_go.Figure())

BECLUST3D_PATH = '/content/beclust3d-public'
sys.path.insert(0, BECLUST3D_PATH)
sys.path.insert(0, os.path.join(BECLUST3D_PATH, 'examples'))

from be3d_local_helper import (
    show_svgs, show_images, plot_residue_dot, plot_ppi_vs_noppi_scatter,
    render_molstar, load_molstar_pdb, color_molstar, chain_values_from_df, edit_yaml_widgets,
)
from be3d_plotly import (
    show_side_by_side, show_stacked, show_picker, plot_hypothesis_qa, plot_violin_by_processed_muttype, plot_score_scatter,
    plot_dendrogram, plot_meta_dendrogram, plot_lfc_lfc3d_scatter, plot_plddt_rsa_scatter,
    plot_domain_barplot, plot_plddt_dis_barplot, plot_enrichment_test,
    plot_meta_score_scatter, plot_meta_lfc_lfc3d_scatter, plot_meta_plddt_rsa_scatter,
    plot_meta_domain_barplot, plot_meta_plddt_dis_barplot,
    COLOR_POS, COLOR_NEG,
)

def run_be3d(yaml_path):
    script = os.path.join(BECLUST3D_PATH, 'examples', 'be3d_local.py')
    # Capture + print explicitly rather than letting the child inherit stdout/stderr --
    # a subprocess's inherited file descriptors don't reliably show up in a notebook
    # cell's own output (Jupyter/Colab capture sys.stdout at the Python level, which a
    # child process's raw fd can bypass), so check=True alone can raise CalledProcessError
    # with no visible clue about what actually went wrong inside be3d_local.py.
    result = subprocess.run([sys.executable, script, yaml_path], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        result.check_returncode()

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

def run_be3d_if_needed(yaml_path, done_marker):
    if os.path.exists(done_marker):
        print(f'[skip] {done_marker} already exists')
    else:
        run_be3d(yaml_path)


# Settings
- Choose which mode to run: monomer, ppi, or blind_target
- Loads that mode's default YAML config
- Example gene: KBTBD4-HDAC1 (PDB 8VOJ), which has data to support all three modes


In [ ]:
# KBTBD4-HDAC1 (8VOJ) supports all three modes: monomer, ppi (via ppi_diff), and blind_target.
# These yaml configs (and the example screens/PDB they point at) were cloned above into
# /content/beclust3d-public/examples/{yaml,data,pdb} -- pipeline outputs go to a separate
# /content/BE3D_example/output/ scratch dir, not into the cloned repo tree.
YAML_DIR = '/content/beclust3d-public/examples/yaml'
MONOMER_YAML = f'{YAML_DIR}/KBTBD4_chain_B_colab.yaml'
PPI_YAML = f'{YAML_DIR}/ppi_diff_KBTBD4_HDAC1_colab.yaml'
BLIND_TARGET_YAML = f'{YAML_DIR}/blind_target_KBTBD4_HDAC1_colab.yaml'

for label, path in [('monomer', MONOMER_YAML), ('ppi (ppi_diff)', PPI_YAML), ('blind_target', BLIND_TARGET_YAML)]:
    cfg = load_yaml(path)
    print(f"--- {label}: mode='{cfg.get('mode')}' ---")
    shown = {k: cfg[k] for k in ('input_gene', 'input_uniprot', 'input_chain', 'output_dir') if k in cfg}
    print(yaml.safe_dump(shown, sort_keys=False))


## Select a mode
- Pick one of monomer / ppi / blind_target below
- Only the section(s) matching the selected mode actually run further down; the others print a skip message

In [ ]:
import ipywidgets as widgets

MODE_OPTIONS = [
    ('Monomer -- single target gene, no PPI partner', 'monomer'),
    ('PPI -- target gene(s) compared with vs. without a PPI partner (mode: ppi_diff)', 'ppi'),
    ('Blind target -- target has no screen data of its own, scored purely from PPI partner(s)', 'blind_target'),
]

mode_selector = widgets.RadioButtons(
    options=MODE_OPTIONS, description='Mode:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='750px'),
)

MODE = mode_selector.value

def _on_mode_change(change):
    global MODE
    MODE = change['new']
    print(f"Selected mode: '{MODE}' -- re-run the config editor and the cells below for this mode.")

mode_selector.observe(_on_mode_change, names='value')
display(mode_selector)
print(f"Selected mode: '{MODE}' -- re-run the config editor and the cells below for this mode.")


## Edit config for the selected mode
- Widgets for the selected mode's YAML fields, each with its own description
- Edits are written back to the YAML file as soon as you change a field, before the pipeline runs


In [ ]:
YAML_BY_MODE = {'monomer': MONOMER_YAML, 'ppi': PPI_YAML, 'blind_target': BLIND_TARGET_YAML}
EDITABLE_KEYS_BY_MODE = {
    'monomer': ['input_gene', 'input_uniprot', 'input_chain', 'screen_dir', 'screens',
                'output_dir', 'user_pdb', 'user_fasta', 'user_dssp',
                'nRandom', 'structure_radius', 'clustering_radius',
                'function_for_lfc', 'function_for_lfc3d', 'function_for_meta'],
    'ppi': ['input_gene', 'input_uniprot', 'input_chain', 'screen_dir', 'screens',
            'output_dir', 'user_pdb', 'user_fasta', 'user_dssp', 'score_type', 'skip_existing',
            'nRandom', 'structure_radius', 'clustering_radius',
            'function_for_lfc', 'function_for_lfc3d', 'function_for_meta'],
    'blind_target': ['input_gene', 'input_uniprot', 'input_chain', 'output_dir',
                      'user_pdb', 'user_fasta', 'user_dssp',
                      'function_for_lfc', 'function_for_lfc3d', 'function_for_meta'],
}

print(f"Editing config for mode '{MODE}' ({YAML_BY_MODE[MODE]}) -- "
      "changes below are written back to the yaml file immediately, picked up the next "
      "time a cell further down runs the pipeline. Nested settings (pthr, database, "
      "mutation_category, qa, ...) aren't exposed here -- edit the yaml file directly for those.")
edit_yaml_widgets(YAML_BY_MODE[MODE], EDITABLE_KEYS_BY_MODE[MODE])


# BE-QA
- Hypothesis-test scatter plots (Kolmogorov-Smirnov and Mann-Whitney statistic vs. -log10(p)), one pair per screen
- Violin plot of the processed per-guide LFC distribution by mutation category, for the selected screen
- In ppi mode, both are shown twice per gene (no-PPI leg, then PPI-mode leg) side by side
- In blind_target mode, only the violin plot is available per partner (partners skip the KS/MW hypothesis test)


In [ ]:
if MODE == 'monomer':
    monomer_cfg = load_yaml(YAML_BY_MODE['monomer'])
    monomer_dir, monomer_gene, monomer_uniprot = monomer_cfg['output_dir'], monomer_cfg['input_gene'], monomer_cfg['input_uniprot']
    run_be3d_if_needed(YAML_BY_MODE['monomer'], os.path.join(monomer_dir, 'RUN_COMPLETED.txt'))

    monomer_screens = [s.strip().split('.')[0] for s in monomer_cfg['screens'].split(',')]

    monomer_hyp_ks = plot_hypothesis_qa(monomer_dir, test='KolmogorovSmirnov')
    monomer_hyp_mw = plot_hypothesis_qa(monomer_dir, test='MannWhitney')

elif MODE == 'ppi':
    ppi_cfg = load_yaml(YAML_BY_MODE['ppi'])
    ppi_root = ppi_cfg['output_dir']
    gene_names = [g.strip() for g in ppi_cfg['input_gene'].split(',')]
    ppi_screens = [s.strip().split('.')[0] for s in ppi_cfg['screens'].split(',')]

    # The QA/violin cells below need the per-gene no_ppi and ppi legs to already exist, so run
    # the pipeline first (mirrors what the BE-Clust3D (PPI) cell does further down). Variant
    # yamls go to the scratch dir, not the cloned repo's yaml dir (see Settings cell comment).
    def run_ppi_diff_pass(score_type):
        variant_cfg = copy.deepcopy(ppi_cfg)
        variant_cfg['score_type'] = score_type
        variant_yaml = os.path.join('/content/BE3D_example', f'_ppi_diff_{score_type}.yaml')
        os.makedirs('/content/BE3D_example', exist_ok=True)
        with open(variant_yaml, 'w') as f:
            yaml.safe_dump(variant_cfg, f)
        run_be3d(variant_yaml)

    run_ppi_diff_pass('LFC3D')

else:
    blind_cfg = load_yaml(YAML_BY_MODE['blind_target'])
    blind_dir = blind_cfg['output_dir']
    blind_gene, blind_chain = blind_cfg['input_gene'], blind_cfg['input_chain']
    blind_partners = blind_cfg['partners']
    blind_tsv = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D.tsv')

    # blind_target's target itself never gets hypothesis_test/screendata (run_blind_target skips
    # both -- it has no screen data of its own). Each partner runs through parse_be_data +
    # prioritize_by_sequence (preprocess_ppi_partner), same as a monomer run, but
    # preprocess_ppi_partner explicitly skips hypothesis_test -- so there's no KS2/MW QA to show
    # even for the partner, just its processed-LFC violin, under
    # {blind_dir}/ppi_partners/{gene}_chain_{chain}/screendata/.
    if not os.path.exists(blind_tsv):
        run_be3d(YAML_BY_MODE['blind_target'])

    partner_by_gene = {p['gene']: p for p in blind_partners}
    partner_names = list(partner_by_gene)
    partner_screens_by_gene = {
        gene: [s.strip().split('.')[0] for s in p['screens'].split(',')]
        for gene, p in partner_by_gene.items()
    }
    all_partner_screens = sorted({s for screens in partner_screens_by_gene.values() for s in screens})

print(f"[done] BE-QA data ready for mode '{MODE}' -- run the cell below to visualize (no pipeline re-run needed to just change a dropdown).")


In [ ]:
if MODE == 'monomer':
    def show_monomer_qa(screen_name):
        print('QA (KS2, MW test, all screens) and processed LFC distribution by mutation category '
                '(violin, post mutation_priority + per-category filtering):')
        violin_fig = plot_violin_by_processed_muttype(monomer_dir, monomer_gene, screen_name)
        show_side_by_side(monomer_hyp_ks, monomer_hyp_mw, violin_fig, width=600, height=400, spacing=0.08)

    interact_manual(show_monomer_qa, screen_name=Dropdown(options=monomer_screens, description='Screen:'));

elif MODE == 'ppi':
    def show_ppi_qa(gene, screen_name):
        noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
        ppi_dir = os.path.join(ppi_root, 'ppi', gene)

        print(f'{gene} -- QA (KS2 test), no-PPI then PPI-mode:')
        show_side_by_side(
            plot_hypothesis_qa(noppi_dir, test='KolmogorovSmirnov'),
            plot_hypothesis_qa(ppi_dir, test='KolmogorovSmirnov'),
            width=600, height=400,
        )
        print(f'{gene} -- QA (MW test), no-PPI then PPI-mode:')
        show_side_by_side(
            plot_hypothesis_qa(noppi_dir, test='MannWhitney'),
            plot_hypothesis_qa(ppi_dir, test='MannWhitney'),
            width=600, height=400,
        )
        print('Processed LFC distribution by mutation category (violin, post mutation_priority + '
                'per-category filtering), no-PPI then PPI-mode:')
        show_side_by_side(
            plot_violin_by_processed_muttype(noppi_dir, gene, screen_name),
            plot_violin_by_processed_muttype(ppi_dir, gene, screen_name),
            width=600, height=400,
        )

    interact_manual(
        show_ppi_qa,
        gene=Dropdown(options=gene_names, description='Gene:'),
        screen_name=Dropdown(options=ppi_screens, description='Screen:'),
    );

else:
    print("Note: blind_target partners skip hypothesis_test (preprocess_ppi_partner), so no KS2/MW "
            "QA plot is available -- showing each partner's processed LFC distribution (violin) instead:")

    def show_blind_qa(partner_gene, screen_name):
        if screen_name not in partner_screens_by_gene[partner_gene]:
            print(f"{partner_gene} has no screen '{screen_name}' -- pick one of {partner_screens_by_gene[partner_gene]}")
            return
        partner_chain = partner_by_gene[partner_gene]['chain']
        partner_dir = os.path.join(blind_dir, 'ppi_partners', f'{partner_gene}_chain_{partner_chain}')

        violin_fig = plot_violin_by_processed_muttype(partner_dir, partner_gene, screen_name)
        if violin_fig is not None:
            display(violin_fig)

    interact_manual(
        show_blind_qa,
        partner_gene=Dropdown(options=partner_names, description='Partner:'),
        screen_name=Dropdown(options=all_partner_screens, description='Screen:'),
    );


# Monomer Mode

## BE-Clust3D
- Residue dot-plots of LFC and LFC3D (positive and negative shown separately), for the selected screen
- LFC vs. LFC3D scatter, highlighting residues with an LFC3D value but no direct LFC value
- pLDDT vs. RSA scatter, LFC3D hit count by protein domain, and hit count by pLDDT-disorder category
- Enrichment test (log2 odds ratio) for pLDDT-disorder category
- Dendrogram of spatial clustering, switchable between LFC/LFC3D and positive/negative


In [ ]:
if MODE == 'monomer':
    DENDRO_OPTIONS = ['LFC positive', 'LFC negative', 'LFC3D positive', 'LFC3D negative']

    def show_monomer_clust3d(screen_name, dendrogram):
        print('Residue dot-plots, LFC (positive, negative):')
        show_side_by_side(
            plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='positive'),
            plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='negative'),
    		width=600, height=400
        )
        print('Residue dot-plots, LFC3D (positive, negative):')
        show_side_by_side(
            plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='positive'),
            plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='negative'),
    		width=600, height=400
        )

        print('LFC vs. LFC3D (residues with LFC3D but no LFC shown in the left strip):')
        fig = plot_lfc_lfc3d_scatter(monomer_dir, monomer_gene, screen_name, width=500, height=400)
        if fig is not None:
            display(fig)

        print('pLDDT vs. RSA / LFC3D hit count by domain / pLDDT-disorder category:')
        show_side_by_side(
            plot_plddt_rsa_scatter(monomer_dir, monomer_gene, screen_name),
            plot_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, screen_name),
            plot_plddt_dis_barplot(monomer_dir, monomer_gene, screen_name),
            height=400, width=600
        )

        print('Enrichment test (pLDDT-disorder, log2 odds ratio):')
        fig = plot_enrichment_test(monomer_dir, monomer_gene, screen_name=screen_name, width=400, height=300)
        if fig is not None:
            display(fig)

        # Dendrogram choice is its own top-level interact() parameter here, not a nested
        # show_picker() (= a second interact()) called from inside this callback -- nesting
        # one interact() inside another flickers and vanishes under Colab's custom widget
        # manager. One flat interact() with both dropdowns never nests an Output inside
        # another, so it stays put.
        score_type, direction = {
            'LFC positive': ('LFC', 'Positive'), 'LFC negative': ('LFC', 'Negative'),
            'LFC3D positive': ('LFC3D', 'Positive'), 'LFC3D negative': ('LFC3D', 'Negative'),
        }[dendrogram]
        print(f'Dendrogram (p<0.05) -- {dendrogram}:')
        fig = plot_dendrogram(monomer_dir, monomer_gene, screen_name, score_type=score_type, direction=direction, height=400)
        if fig is not None:
            display(fig)
        else:
            print('[not available]')

    interact_manual(
        show_monomer_clust3d,
        screen_name=Dropdown(options=monomer_screens, description='Screen:'),
        dendrogram=Dropdown(options=DENDRO_OPTIONS, description='Dendrogram:'),
    );
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-Clust3D (monomer).")


## BE-MetaClust3D
- Only shown when the gene has multiple screens (otherwise there's nothing to meta-aggregate)
- Same plots as BE-Clust3D above, but computed on the meta-aggregated score across all screens instead of one screen at a time (meta residue dot-plots, meta-LFC vs. meta-LFC3D scatter, pLDDT/RSA and domain/disorder breakdowns, enrichment test, meta dendrogram)


In [ ]:
if MODE == 'monomer':
    monomer_func_meta = monomer_cfg['function_for_meta']

    if len(monomer_screens) > 1:
        print('Meta residue dot-plots, meta-LFC (positive, negative):')
        show_side_by_side(
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='positive'),
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='negative'),
    		width=600, height=400
        )
        print('Meta residue dot-plots, meta-LFC3D (positive, negative):')
        show_side_by_side(
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='positive'),
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='negative'),
    		width=600, height=400
        )

        print('meta-LFC vs. meta-LFC3D (residues with meta-LFC3D but no meta-LFC shown in the left strip):')
        fig = plot_meta_lfc_lfc3d_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, width=500, height=400)
        if fig is not None:
            display(fig)

        print('pLDDT vs. RSA / Meta LFC3D hit count by domain / pLDDT-disorder category:')
        show_side_by_side(
            plot_meta_plddt_rsa_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
            plot_meta_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, function_for_meta=monomer_func_meta),
            plot_meta_plddt_dis_barplot(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
            width=600, height=400
        )

        print('Enrichment test (pLDDT-disorder, log2 odds ratio):')
        fig = plot_enrichment_test(monomer_dir, monomer_gene, screen_name=None, width=400, height=300)
        if fig is not None:
            display(fig)

        print('Meta dendrogram (p<0.05) -- pick a score type / direction:')
        show_picker({
            'Meta LFC positive': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='Positive', height=400),
            'Meta LFC negative': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='Negative', height=400),
            'Meta LFC3D positive': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='Positive', height=400),
            'Meta LFC3D negative': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='Negative', height=400),
        }, description='Dendrogram:')
    else:
        print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-MetaClust3D (monomer).")


# PPI mode

## BE-Clust3D
- For the selected gene/screen: residue dot-plots of LFC3D (positive and negative), no-PPI leg then PPI-mode leg side by side
- No-PPI LFC3D (x-axis) vs. PPI-mode LFC3D (y-axis) scatter, to see how each residue's score shifts between the two
- LFC vs. LFC3D scatter, pLDDT vs. RSA scatter, and LFC3D hit count by pLDDT-disorder category, each shown no-PPI then PPI-mode
- Enrichment test (pLDDT-disorder), no-PPI then PPI-mode
- Dendrogram, switchable between no-PPI/PPI-mode and positive/negative


In [ ]:
if MODE == 'ppi':
    ppi_cfg = load_yaml(PPI_YAML)
    ppi_root = ppi_cfg['output_dir']
    gene_names = [g.strip() for g in ppi_cfg['input_gene'].split(',')]
    chain_list = [c.strip() for c in ppi_cfg['input_chain'].split(',')]
    ppi_screens = [s.strip().split('.')[0] for s in ppi_cfg['screens'].split(',')]

    # mode: ppi_diff runs the PPI leg (mode: complex) and no-PPI leg (mode: monomer, per gene)
    # once, then merges -- skip_existing makes each pass a no-op for the pipeline legs once the
    # first pass has run them. score_type controls only the (cheap) merge step: 'LFC3D' produces
    # one merged TSV+PDB set per screen; 'Meta_LFC3D' produces the meta-aggregated one (needed by
    # the BE-MetaClust3D and Merged-results sections below).
    def run_ppi_diff_pass(score_type):
        variant_cfg = copy.deepcopy(ppi_cfg)
        variant_cfg['score_type'] = score_type
        variant_yaml = os.path.join('/content/BE3D_example', f'_ppi_diff_{score_type}.yaml')
        os.makedirs('/content/BE3D_example', exist_ok=True)
        with open(variant_yaml, 'w') as f:
            yaml.safe_dump(variant_cfg, f)
        run_be3d(variant_yaml)

    run_ppi_diff_pass('LFC3D')
    run_ppi_diff_pass('Meta_LFC3D')

    ppi_func_meta = ppi_cfg['function_for_meta']
    ppi_uniprot_by_gene = dict(zip(gene_names, [u.strip() for u in ppi_cfg['input_uniprot'].split(',')]))
    DENDRO_OPTIONS = ['no-PPI positive', 'PPI-mode positive', 'no-PPI negative', 'PPI-mode negative']

    print("[done] PPI-diff data ready -- run the cell below to visualize (no pipeline re-run needed to just change a dropdown).")
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-Clust3D (ppi) setup.")


In [ ]:
if MODE == 'ppi':
    def show_ppi_clust3d(gene, screen_name, dendrogram):
        chain = dict(zip(gene_names, chain_list))[gene]
        uniprot = ppi_uniprot_by_gene[gene]
        noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
        ppi_dir = os.path.join(ppi_root, 'ppi', gene)

        print(f'{gene} (chain {chain}) -- residue dot-plots, LFC3D positive -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_score_scatter(noppi_dir, gene, screen_name, score_type='LFC3D', direction='positive'),
            plot_score_scatter(ppi_dir, gene, screen_name, score_type='LFC3D', direction='positive'),
    		width=600, height=400
        )
        print('residue dot-plots, LFC3D negative -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_score_scatter(noppi_dir, gene, screen_name, score_type='LFC3D', direction='negative'),
            plot_score_scatter(ppi_dir, gene, screen_name, score_type='LFC3D', direction='negative'),
    		width=600, height=400
        )

        print('no-PPI LFC3D (x) vs. PPI-mode LFC3D (y):')
        df_screen = pd.read_csv(os.path.join(ppi_root, f'ppi_vs_noppi_{screen_name}.tsv'), sep='\t')
        plot_ppi_vs_noppi_scatter(df_screen[df_screen['gene'] == gene], 'LFC3D', width=500, height=400)

        print('LFC vs. LFC3D -- no-PPI, then PPI-mode (residues with LFC3D but no LFC shown in each left strip):')
        show_side_by_side(
            plot_lfc_lfc3d_scatter(noppi_dir, gene, screen_name),
            plot_lfc_lfc3d_scatter(ppi_dir, gene, screen_name),
    		width=600, height=400
        )

        print('pLDDT vs. RSA -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_plddt_rsa_scatter(noppi_dir, gene, screen_name),
            plot_plddt_rsa_scatter(ppi_dir, gene, screen_name),
            width=600, height=400
        )

        print('LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_plddt_dis_barplot(noppi_dir, gene, screen_name),
            plot_plddt_dis_barplot(ppi_dir, gene, screen_name),
            width=600, height=400
        )

        print('Enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_enrichment_test(noppi_dir, gene, screen_name=screen_name),
            plot_enrichment_test(ppi_dir, gene, screen_name=screen_name),
            width=600, height=400
        )

        # Dendrogram choice is its own top-level interact() parameter, not a nested
        # show_picker() (= a second interact()) called from inside this callback -- see the
        # BE-Clust3D (monomer) cell above for why nesting one interact() inside another
        # flickers and vanishes under Colab's custom widget manager.
        dendro_dir, direction = {
            'no-PPI positive': (noppi_dir, 'Positive'), 'PPI-mode positive': (ppi_dir, 'Positive'),
            'no-PPI negative': (noppi_dir, 'Negative'), 'PPI-mode negative': (ppi_dir, 'Negative'),
        }[dendrogram]
        print(f'LFC3D dendrogram (p<0.05) -- {dendrogram}:')
        fig = plot_dendrogram(dendro_dir, gene, screen_name, score_type='LFC3D', direction=direction, height=400)
        if fig is not None:
            display(fig)
        else:
            print('[not available]')

    interact_manual(
        show_ppi_clust3d,
        gene=Dropdown(options=gene_names, description='Gene:'),
        screen_name=Dropdown(options=ppi_screens, description='Screen:'),
        dendrogram=Dropdown(options=DENDRO_OPTIONS, description='Dendrogram:'),
    );
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-Clust3D (ppi).")


## BE-MetaClust3D
- Only shown when the gene has multiple screens
- Same comparisons as BE-Clust3D (ppi) above -- no-PPI vs. PPI-mode -- but on the meta-aggregated meta-LFC3D score instead of a single screen


In [ ]:
if MODE == 'ppi':
    if len(ppi_screens) > 1:
        df_meta = pd.read_csv(os.path.join(ppi_root, 'ppi_vs_noppi_Meta_LFC3D.tsv'), sep='\t')
        DENDRO_OPTIONS = ['no-PPI positive', 'PPI-mode positive', 'no-PPI negative', 'PPI-mode negative']

        def show_ppi_metaclust3d(gene, dendrogram):
            chain = dict(zip(gene_names, chain_list))[gene]
            noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
            ppi_dir = os.path.join(ppi_root, 'ppi', gene)

            print(f'{gene} (chain {chain}) -- meta residue dot-plots, meta-LFC3D positive -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_meta_score_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='positive'),
                plot_meta_score_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='positive'),
                width=600, height=400
            )
            print('meta residue dot-plots, meta-LFC3D negative -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_meta_score_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='negative'),
                plot_meta_score_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='negative'),
                width=600, height=400
            )

            print('no-PPI meta-LFC3D (x) vs. PPI-mode meta-LFC3D (y):')
            plot_ppi_vs_noppi_scatter(df_meta[df_meta['gene'] == gene], 'meta-LFC3D', width=500, height=400)

            print('meta-LFC vs. meta-LFC3D -- no-PPI, then PPI-mode (residues with meta-LFC3D but no meta-LFC shown in each left strip):')
            show_side_by_side(
                plot_meta_lfc_lfc3d_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta),
                plot_meta_lfc_lfc3d_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta),
                width=500, height=500
            )

            print('pLDDT vs. RSA -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_meta_plddt_rsa_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta),
                plot_meta_plddt_rsa_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta),
                width=600, height=400
            )

            print('Meta LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_meta_plddt_dis_barplot(noppi_dir, gene, function_for_meta=ppi_func_meta),
                plot_meta_plddt_dis_barplot(ppi_dir, gene, function_for_meta=ppi_func_meta),
                width=600, height=400
            )

            print('Meta enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_enrichment_test(noppi_dir, gene, screen_name=None),
                plot_enrichment_test(ppi_dir, gene, screen_name=None),
                width=600, height=400
            )

            # Dendrogram choice is its own top-level interact() parameter, not a nested
            # show_picker() (= a second interact()) called from inside this callback -- see
            # the BE-Clust3D (monomer) cell for why nesting one interact() inside another
            # flickers and vanishes under Colab's custom widget manager.
            dendro_dir, direction = {
                'no-PPI positive': (noppi_dir, 'Positive'), 'PPI-mode positive': (ppi_dir, 'Positive'),
                'no-PPI negative': (noppi_dir, 'Negative'), 'PPI-mode negative': (ppi_dir, 'Negative'),
            }[dendrogram]
            print(f'Meta LFC3D dendrogram (p<0.05) -- {dendrogram}:')
            fig = plot_meta_dendrogram(dendro_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction=direction, height=400)
            if fig is not None:
                display(fig)
            else:
                print('[not available]')

        interact_manual(
            show_ppi_metaclust3d,
            gene=Dropdown(options=gene_names, description='Gene:'),
            dendrogram=Dropdown(options=DENDRO_OPTIONS, description='Dendrogram:'),
        );
    else:
        print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-MetaClust3D (ppi).")


## Merged results (meta-LFC3D)
- Table of the top 10 residues ranked by |delta meta-LFC3D| (PPI-mode minus no-PPI)
- One Molstar structure viewer with a dropdown to switch between three colorings: no-PPI meta-LFC3D, PPI-mode meta-LFC3D, and their delta (-2 to +2, white at 0)
- The top 10 |delta| residues from the table above are highlighted as spheres in every view


In [ ]:
if MODE == 'ppi':
    df_merged = pd.read_csv(os.path.join(ppi_root, 'ppi_vs_noppi_Meta_LFC3D.tsv'), sep='\t')
    df_merged_sorted = df_merged.reindex(df_merged['delta_score'].abs().sort_values(ascending=False).index)

    print('Top 10 residues by |delta meta-LFC3D| (PPI - no-PPI):')
    top10 = df_merged_sorted.head(10)
    display(top10[['gene', 'chain', 'unipos', 'unires', 'noppi_score', 'ppi_score', 'delta_score']])
    top10_unipos = top10['unipos'].tolist()
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping PPI merged results.")


In [ ]:
if MODE == 'ppi':
    base_pdb = os.path.join(ppi_root, 'ppi', gene_names[0], 'sequence_structure')
    base_pdb = os.path.join(base_pdb, [f for f in os.listdir(base_pdb) if f.endswith('_processed.pdb')][0])

    merged_views = {
        'No-PPI meta-LFC3D': chain_values_from_df(df_merged, 'noppi_score'),
        'PPI-mode meta-LFC3D': chain_values_from_df(df_merged, 'ppi_score'),
        'Delta (PPI - no-PPI) meta-LFC3D': chain_values_from_df(df_merged, 'delta_score'),
    }

    print('Structure colored by the selected view (spheres = the top 10 |delta| residues from the table above):')
    merged_widget = PDBeMolstar(hide_water=True, height='333px')
    load_molstar_pdb(merged_widget, base_pdb)
    display(merged_widget)

    def show_merged_view(view_name):
        color_molstar(merged_widget, merged_views[view_name], vmax=2.0, highlight_top_n=10)

    interact_manual(show_merged_view, view_name=Dropdown(options=list(merged_views), description='View:'));
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping PPI merged results (structure view).")


# Blind target mode
- Table of residues that received a blind LFC3D value (uses the meta-aggregated column if the target has multiple partner screens, otherwise the single screen's column)
- Residue dot-plot of that same signed blind LFC3D value
- Molstar 3D structure viewer, colored by the selected view (Negative / Positive / Overall); blue marks positive values, red marks negative ones


In [ ]:
if MODE == 'blind_target':
    blind_cfg = load_yaml(YAML_BY_MODE['blind_target'])
    blind_dir = blind_cfg['output_dir']
    blind_gene, blind_chain = blind_cfg['input_gene'], blind_cfg['input_chain']
    blind_tsv = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D.tsv')

    if not os.path.exists(blind_tsv):
        run_be3d(YAML_BY_MODE['blind_target'])

    df_blind = pd.read_csv(blind_tsv, sep='\t')

    # use the meta-aggregated column if there's more than one partner screen, else the single screen's
    screen_names = [c[:-len('_LFC3D_blind_overall')] for c in df_blind.columns if c.endswith('_LFC3D_blind_overall')]
    if 'Meta_LFC3D_blind_overall' in df_blind.columns:
        neg_col, pos_col, overall_col = 'Meta_LFC3D_blind_neg', 'Meta_LFC3D_blind_pos', 'Meta_LFC3D_blind_overall'
    else:
        screen_name = screen_names[0]
        neg_col, pos_col, overall_col = f'{screen_name}_LFC3D_blind_neg', f'{screen_name}_LFC3D_blind_pos', f'{screen_name}_LFC3D_blind_overall'

    df_blind_hits = df_blind[(df_blind[neg_col] != '-') | (df_blind[pos_col] != '-')]
    print(f'{blind_gene} (chain {blind_chain}) -- {len(df_blind_hits)}/{len(df_blind)} residues have a blind LFC3D value:')
    display(df_blind_hits[['unipos', 'unires', 'chain', neg_col, pos_col, overall_col]])
else:
    print(f"[skipped] mode is '{MODE}', not 'blind_target' -- skipping blind-target results.")


In [ ]:
if MODE == 'blind_target':
    print('Residue dot-plot (signed value: negative or positive column, whichever is set):')
    signed = pd.to_numeric(df_blind[neg_col].replace('-', pd.NA), errors='coerce')
    signed = signed.fillna(pd.to_numeric(df_blind[pos_col].replace('-', pd.NA), errors='coerce'))
    df_blind_signed = df_blind.copy()
    df_blind_signed['_signed_blind_LFC3D'] = signed
    plot_residue_dot(df_blind_signed, '_signed_blind_LFC3D', f'{blind_gene} blind LFC3D')


In [ ]:
if MODE == 'blind_target':
    overall_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_overall.pdb')
    pos_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_pos.pdb')
    neg_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_neg.pdb')
    # run_blind_target always writes all three PDBs together (same coordinates, different
    # B-factors baked in) when user_pdb is set, so which one is loaded as the base structure
    # doesn't matter -- only the color_data changes per dropdown selection below.
    blind_base_pdb = overall_pdb if os.path.exists(overall_pdb) else (pos_pdb if os.path.exists(pos_pdb) else neg_pdb)

    def _blind_chain_values(col):
        values = pd.to_numeric(df_blind[col].replace('-', pd.NA), errors='coerce')
        return {blind_chain: {int(p): float(v) for p, v in zip(df_blind['unipos'], values) if pd.notna(v)}}

    blind_views = {
        'Negative': _blind_chain_values(neg_col),
        'Positive': _blind_chain_values(pos_col),
        'Overall': _blind_chain_values(overall_col),
    }

    blind_widget = PDBeMolstar(hide_water=True, height='333px')
    load_molstar_pdb(blind_widget, blind_base_pdb)
    display(blind_widget)

    def show_blind_view(view_name):
        color_molstar(blind_widget, blind_views[view_name], vmax=2.0, highlight_top_n=10)

    interact_manual(show_blind_view, view_name=Dropdown(options=list(blind_views), description='View:'));
else:
    print(f"[skipped] mode is '{MODE}', not 'blind_target' -- skipping blind-target results (structure view).")
